# Fanout Variance Analysis
Measure how search value variance decreases with fanout. For each (game stage, depth, fanout), run many trials from the same state.

In [7]:
import numpy as np
import matplotlib.pyplot as plt
import time
from game import GameConfig, sample_transitions
from engine import Engine

In [8]:
# --- CONFIG ---
game_config = GameConfig.small()
TRIALS = 100
DEPTHS = [1, 2, 3]
FANOUTS = [1, 3, 5, 10, 15, 30]
ROUNDS = [1, 3, 5]  # rounds into the game to test at, i.e. 5x number of objs removed

In [9]:
combos = len(ROUNDS) * len(DEPTHS) * len(FANOUTS)
print(f"Trials: {TRIALS}, depths: {DEPTHS}, fanouts: {FANOUTS}")
print(f"Game stages: rounds {ROUNDS} of {game_config.num_rounds}")
print(f"Total combos: {combos} ({combos * TRIALS} search calls)")

# Warmup
Engine(1, 5, game_config).search_value(game_config.init_stashed, game_config.init_pool)
print("Ready.")

Trials: 100, depths: [1, 2, 3], fanouts: [1, 3, 5, 10, 15, 30]
Game stages: rounds [1, 3, 5] of 6
Total combos: 54 (5400 search calls)
Ready.


In [11]:
def get_state_at_round(config, target_round):
    s, r = config.init_stashed, config.init_pool
    for _ in range(target_round):
        if sum(r) < config.draw_size:
            break
        transitions, _, _ = sample_transitions(s, r, config)
        s, r = transitions[np.random.randint(len(transitions))]
    return s, r

# Get fixed states
states = {}
for r in ROUNDS:
    stashed, remaining = get_state_at_round(game_config, r)
    deck = sum(remaining)
    if deck < game_config.draw_size:
        print(f"Round {r}: game over (deck={deck}), skipping")
        continue
    states[r] = (stashed, remaining)
    print(f"Round {r}/{game_config.num_rounds}: deck={deck}, stashed={stashed}")

Round 1/6: deck=25, stashed=(1, 0, 0, 0, 1)
Round 3/6: deck=15, stashed=(1, 4, 1, 1, 0)
Round 5/6: deck=5, stashed=(3, 3, 0, 2, 1)


In [12]:
# Run trials
results = {}
for r, (stashed, remaining) in states.items():
    for depth in DEPTHS:
        key = (r, depth)
        results[key] = {}
        for fanout in FANOUTS:
            t0 = time.time()
            vals = np.array([
                Engine(depth, fanout, config).search_value(stashed, remaining)
                for _ in range(TRIALS)
            ])
            elapsed = time.time() - t0
            results[key][fanout] = {"std": np.std(vals), "mean": np.mean(vals), "values": vals}
            print(f"  round={r}, d={depth}, f={fanout:3d}: "
                  f"mean={np.mean(vals):6.1f}, std={np.std(vals):5.2f}  ({elapsed:.1f}s)")
        print()

  round=1, d=1, f=  1: mean=   3.0, std= 6.81  (0.0s)
  round=1, d=1, f=  3: mean=   4.7, std= 4.21  (0.0s)
  round=1, d=1, f=  5: mean=   4.4, std= 3.49  (0.0s)
  round=1, d=1, f= 10: mean=   4.5, std= 2.36  (0.0s)
  round=1, d=1, f= 15: mean=   3.7, std= 1.86  (0.0s)
  round=1, d=1, f= 30: mean=   4.1, std= 1.53  (0.0s)

  round=1, d=2, f=  1: mean=  18.4, std= 6.46  (0.0s)
  round=1, d=2, f=  3: mean=  16.8, std= 2.77  (0.0s)
  round=1, d=2, f=  5: mean=  16.5, std= 2.13  (0.0s)
  round=1, d=2, f= 10: mean=  15.4, std= 1.28  (0.0s)
  round=1, d=2, f= 15: mean=  15.2, std= 0.97  (0.1s)
  round=1, d=2, f= 30: mean=  15.0, std= 0.69  (0.3s)

  round=1, d=3, f=  1: mean=  43.6, std=10.61  (0.0s)
  round=1, d=3, f=  3: mean=  32.2, std= 3.03  (0.0s)
  round=1, d=3, f=  5: mean=  29.5, std= 1.75  (0.2s)
  round=1, d=3, f= 10: mean=  27.6, std= 1.20  (1.2s)
  round=1, d=3, f= 15: mean=  26.9, std= 0.89  (3.9s)
  round=1, d=3, f= 30: mean=  26.3, std= 0.45  (30.9s)

  round=3, d=1, f=  1: m

In [ ]:
# Std dev vs fanout
fig, axes = plt.subplots(1, len(states), figsize=(5 * len(states), 4), squeeze=False, sharey=True)

for col, r in enumerate(sorted(states.keys())):
    ax = axes[0, col]
    deck = sum(states[r][1])
    ax.set_title(f"Round {r}/{game_config.num_rounds} (deck={deck})")
    ax.set_xlabel("Fanout")
    if col == 0:
        ax.set_ylabel("Std dev of search value")

    for depth in DEPTHS:
        key = (r, depth)
        if key not in results:
            continue
        fanouts = sorted(results[key].keys())
        stds = [results[key][f]["std"] for f in fanouts]
        ax.plot(fanouts, stds, "o-", label=f"depth={depth}")

    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Box plots of full distributions
fig, axes = plt.subplots(len(DEPTHS), len(states),
                         figsize=(5 * len(states), 3.5 * len(DEPTHS)), squeeze=False)

for col, r in enumerate(sorted(states.keys())):
    deck = sum(states[r][1])
    for row, depth in enumerate(DEPTHS):
        ax = axes[row, col]
        key = (r, depth)
        if key not in results:
            continue

        fanouts = sorted(results[key].keys())
        data = [results[key][f]["values"] for f in fanouts]
        ax.boxplot(data, labels=[str(f) for f in fanouts])

        if row == 0:
            ax.set_title(f"Round {r} (deck={deck})")
        if row == len(DEPTHS) - 1:
            ax.set_xlabel("Fanout")
        if col == 0:
            ax.set_ylabel(f"Value (d={depth})")
        ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()